# 🚧 Pothole Detection - YOLOv8 Training (Google Colab)

**Sistem Deteksi Jalan Berlubang** - Training notebook dengan GPU T4 gratis!

### Cara pakai:
1. Buka di Google Colab
2. Pastikan Runtime → **Change runtime type → GPU (T4)**
3. Jalankan semua cell dari atas ke bawah
4. Download model `best.pt` di akhir
5. Taruh di folder `models/` project kamu

---

## 1️⃣ Cek GPU & Install Dependencies

In [ ]:
# Cek GPU yang tersedia
!nvidia-smi
print()

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Install ultralytics & kagglehub
!pip install ultralytics kagglehub -q

from ultralytics import YOLO
import kagglehub
print('\n✅ Dependencies installed!')

## 2️⃣ Download Dataset dari Kaggle

In [ ]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path

# Download dataset
print('📥 Downloading pothole dataset from Kaggle...')
dataset_path = kagglehub.dataset_download('chitholian/annotated-potholes-dataset')
print(f'✅ Downloaded to: {dataset_path}')

# Count files
img_dir = Path(dataset_path) / 'annotated-images'
jpg_count = len(list(img_dir.glob('*.jpg')))
xml_count = len(list(img_dir.glob('*.xml')))
print(f'📊 Found: {jpg_count} images, {xml_count} annotations')

## 3️⃣ Convert ke Format YOLOv8

In [ ]:
# Setup directories
DATASET_DIR = Path('/content/pothole_dataset')

for split in ['train', 'valid', 'test']:
    (DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

def xml_to_yolo(xml_path, img_w, img_h):
    """Convert Pascal VOC XML to YOLO format."""
    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    labels = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)
        x_center = max(0, min(1, ((xmin + xmax) / 2.0) / img_w))
        y_center = max(0, min(1, ((ymin + ymax) / 2.0) / img_h))
        width = max(0, min(1, (xmax - xmin) / img_w))
        height = max(0, min(1, (ymax - ymin) / img_h))
        labels.append(f'0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')
    return labels

# Find all image-XML pairs
pairs = []
for img_file in sorted(img_dir.glob('*.jpg')):
    xml_file = img_file.with_suffix('.xml')
    if xml_file.exists():
        pairs.append((img_file, xml_file))

print(f'Found {len(pairs)} annotated image pairs')

# Shuffle and split: 80/15/5
random.seed(42)
random.shuffle(pairs)
n = len(pairs)
train_end = int(n * 0.80)
valid_end = int(n * 0.95)

split_map = {
    'train': pairs[:train_end],
    'valid': pairs[train_end:valid_end],
    'test': pairs[valid_end:],
}

# Convert each pair
counts = {}
for split, split_pairs in split_map.items():
    count = 0
    for idx, (img_file, xml_file) in enumerate(split_pairs):
        try:
            tree = ET.parse(str(xml_file))
            root = tree.getroot()
            size_elem = root.find('size')
            img_w = float(size_elem.find('width').text)
            img_h = float(size_elem.find('height').text)
            if img_w <= 0 or img_h <= 0:
                continue
            labels = xml_to_yolo(xml_file, img_w, img_h)
            if not labels:
                continue
            new_name = f'pothole_{split}_{idx:05d}'
            shutil.copy2(img_file, DATASET_DIR / split / 'images' / f'{new_name}.jpg')
            with open(DATASET_DIR / split / 'labels' / f'{new_name}.txt', 'w') as f:
                f.write('\n'.join(labels) + '\n')
            count += 1
        except Exception as e:
            pass
    counts[split] = count
    print(f'  {split}: {count} images')

print(f'\n✅ Total: {sum(counts.values())} images converted!')

In [ ]:
# Create data.yaml
data_yaml = f"""# Pothole Detection Dataset
path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: pothole
"""

with open(DATASET_DIR / 'data.yaml', 'w') as f:
    f.write(data_yaml)

print('✅ data.yaml created!')
print()

# Preview beberapa gambar
import matplotlib.pyplot as plt
import cv2

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Sample Training Images', fontsize=16)

train_imgs = sorted((DATASET_DIR / 'train' / 'images').glob('*.jpg'))[:4]
for ax, img_path in zip(axes, train_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4️⃣ Training YOLOv8 🚀

Training dengan **GPU T4** — estimasi ~15-30 menit untuk 100 epochs!

Model options:
- `yolov8n.pt` → Nano (tercepat, cocok real-time)
- `yolov8s.pt` → Small (balance speed/accuracy)
- `yolov8m.pt` → Medium (lebih akurat)

In [ ]:
# ========================================
# 🧠 TRAINING CONFIG - Sesuaikan di sini!
# ========================================

MODEL = 'yolov8n.pt'   # Ganti ke yolov8s.pt atau yolov8m.pt untuk akurasi lebih tinggi
EPOCHS = 100            # Jumlah epoch (100 cukup bagus)
BATCH_SIZE = 16         # Batch size (16 optimal untuk T4)
IMG_SIZE = 640          # Image size

# ========================================

# Load pre-trained model
model = YOLO(MODEL)

# Train!
results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=0,              # GPU
    project='/content/runs',
    name='pothole_detector',
    patience=30,           # Early stopping
    save=True,
    save_period=10,
    cache='ram',           # Cache di RAM (Colab punya 12GB+)
    workers=2,
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    verbose=True,
    seed=42,
    plots=True,
    # Data Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

print()
print('🎉 TRAINING COMPLETE!')

## 5️⃣ Evaluasi Model

In [ ]:
# Load best model
best_model = YOLO('/content/runs/pothole_detector/weights/best.pt')

# Validate
metrics = best_model.val(data=str(DATASET_DIR / 'data.yaml'))

print()
print('=' * 50)
print('📊 VALIDATION RESULTS')
print('=' * 50)
print(f'  mAP50     : {metrics.box.map50:.4f}')
print(f'  mAP50-95  : {metrics.box.map:.4f}')
print(f'  Precision : {metrics.box.mp:.4f}')
print(f'  Recall    : {metrics.box.mr:.4f}')
print('=' * 50)

In [ ]:
# Tampilkan training curves
from IPython.display import Image, display

results_dir = Path('/content/runs/pothole_detector')

# Training results
if (results_dir / 'results.png').exists():
    print('📈 Training Curves:')
    display(Image(filename=str(results_dir / 'results.png'), width=900))

# Confusion matrix
if (results_dir / 'confusion_matrix.png').exists():
    print('\n📊 Confusion Matrix:')
    display(Image(filename=str(results_dir / 'confusion_matrix.png'), width=600))

# Validation predictions
if (results_dir / 'val_batch0_pred.png').exists():
    print('\n🔍 Validation Predictions:')
    display(Image(filename=str(results_dir / 'val_batch0_pred.png'), width=900))

## 6️⃣ Test Deteksi pada Gambar

In [ ]:
# Test pada beberapa gambar test set
import matplotlib.pyplot as plt
import cv2

test_images = sorted((DATASET_DIR / 'test' / 'images').glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('🔍 Pothole Detection Results', fontsize=18, fontweight='bold')

for ax, img_path in zip(axes.flat, test_images):
    # Run detection
    results = best_model(str(img_path), conf=0.4)
    
    # Get annotated image
    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    # Count detections
    n_det = len(results[0].boxes)
    
    ax.imshow(annotated)
    ax.set_title(f'Detections: {n_det}', fontsize=12,
                 color='red' if n_det > 0 else 'green')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7️⃣ Download Model 📦

Download `best.pt` lalu taruh di folder `models/` project-mu:
```
PROJEK JALAN BERLUBANG/
  models/
    best.pt  ← taruh di sini!
```

Lalu jalankan app:
```bash
python app/main.py
```

In [ ]:
# Copy best model ke lokasi yang mudah diakses
import shutil

src = Path('/content/runs/pothole_detector/weights/best.pt')
dst = Path('/content/best.pt')

if src.exists():
    shutil.copy2(src, dst)
    size_mb = dst.stat().st_size / (1024 * 1024)
    print(f'✅ Model saved: {dst}')
    print(f'📦 Size: {size_mb:.1f} MB')
    print()
    print('📥 Download model dari panel Files (kiri) → klik kanan best.pt → Download')
    print('   Atau jalankan cell di bawah untuk auto-download!')
else:
    print('❌ Model not found! Training mungkin belum selesai.')

In [ ]:
# Auto-download model ke komputer kamu
from google.colab import files

if Path('/content/best.pt').exists():
    files.download('/content/best.pt')
    print('\n🎉 Model downloaded! Taruh di folder models/ project-mu.')
    print('   Lalu jalankan: python app/main.py')
else:
    print('❌ best.pt not found!')

---
## ✅ Selesai!

### Langkah selanjutnya:
1. Download file `best.pt`
2. Taruh di `PROJEK JALAN BERLUBANG/models/best.pt`
3. Jalankan `python app/main.py` atau double-click `run.bat`
4. Arahkan webcam ke gambar jalan berlubang
5. Sistem akan bunyi **TIT TIT TIT** saat ada lubang! 🚧

---
*Powered by YOLOv8 + Ultralytics*